# CoT vs Coconut Energy Study — Scaled Run (1,000 train / 200 eval)

Reproducibility study of Coconut (Hao et al., 2024) vs standard Chain-of-Thought,
comparing training energy, inference token count, and accuracy on ProsQA.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run cells top to bottom. Each training cell checkpoints to Google Drive every
1,000 steps and will auto-resume from the last checkpoint if interrupted —
just re-run the same cell.


In [ ]:
# Cell 1 — Install dependencies
!pip install -q torch transformers accelerate tqdm


In [ ]:
# Cell 2 — Mount Drive and set up paths
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/coconut_research_scaled"
DATA_DIR = f"{BASE}/data"
BASELINE_DIR = f"{BASE}/baseline_cot_model"
COCONUT_DIR = f"{BASE}/coconut_model"
CKPT_DIR = f"{BASE}/checkpoints"
LOG_DIR = f"{BASE}/logs"
for d in [BASE, DATA_DIR, BASELINE_DIR, COCONUT_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")


In [ ]:
# Cell 3 — Download ProsQA directly from the official Coconut repo and subsample
import json, random, urllib.request

RAW_TRAIN_URL = "https://raw.githubusercontent.com/facebookresearch/coconut/main/data/prosqa_train.json"
RAW_TEST_URL  = "https://raw.githubusercontent.com/facebookresearch/coconut/main/data/prosqa_test.json"

N_TRAIN = 1000
N_EVAL  = 200
SEED    = 42

def download_json(url, dest):
    if not os.path.exists(dest):
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
    with open(dest) as f:
        return json.load(f)

raw_train_path = f"{DATA_DIR}/prosqa_train_raw.json"
raw_test_path  = f"{DATA_DIR}/prosqa_test_raw.json"

train_data = download_json(RAW_TRAIN_URL, raw_train_path)
test_data  = download_json(RAW_TEST_URL, raw_test_path)
print(f"Full train set: {len(train_data)} examples | Full test set: {len(test_data)} examples")

random.seed(SEED)
random.shuffle(train_data)
random.shuffle(test_data)

train_subset = train_data[:N_TRAIN]
eval_subset  = test_data[:N_EVAL]   # held-out test file, never seen in training

def write_baseline(rows, path):
    with open(path, "w") as f:
        for row in rows:
            out = {"question": row["question"], "reasoning": " ".join(row["steps"]), "answer": row["answer"]}
            f.write(json.dumps(out) + "\n")

def write_coconut(rows, path):
    with open(path, "w") as f:
        for row in rows:
            out = {"question": row["question"], "reasoning_steps": row["steps"], "answer": row["answer"]}
            f.write(json.dumps(out) + "\n")

write_baseline(train_subset, f"{DATA_DIR}/train_baseline.jsonl")
write_baseline(eval_subset,  f"{DATA_DIR}/eval_baseline.jsonl")
write_coconut(train_subset,  f"{DATA_DIR}/train_coconut.jsonl")
write_coconut(eval_subset,   f"{DATA_DIR}/eval_coconut.jsonl")

print(f"Wrote {len(train_subset)} train / {len(eval_subset)} eval examples (shuffled, seed={SEED}).")


In [ ]:
# Cell 4 — Energy logger (T4 GPU TDP-based estimate)
import time, platform
from dataclasses import dataclass, field

# T4 rated max power draw (NVIDIA spec: 70W). This is a documented estimate,
# not a live sensor reading -- state this explicitly in your methods section.
REFERENCE_TDP_WATTS = {
    "colab_t4": 70,
    "colab_cpu": 25,
}
DEFAULT_PUE = 1.0

@dataclass
class EnergyLogger:
    device_name: str
    device_tdp_watts: float
    pue: float = DEFAULT_PUE
    _start_time: float = field(default=None, init=False, repr=False)
    _end_time: float = field(default=None, init=False, repr=False)
    checkpoints: list = field(default_factory=list, init=False)

    def start(self):
        self._start_time = time.time()
        self.checkpoints.append({"event": "start", "t": self._start_time})
        print(f"[EnergyLogger] Started on {self.device_name} (TDP={self.device_tdp_watts}W, PUE={self.pue})", flush=True)

    def checkpoint(self, label):
        self.checkpoints.append({"event": label, "t": time.time()})

    def stop(self):
        self._end_time = time.time()
        self.checkpoints.append({"event": "stop", "t": self._end_time})

    @property
    def elapsed_hours(self):
        if self._start_time is None:
            return 0.0
        return ((self._end_time or time.time()) - self._start_time) / 3600.0

    @property
    def energy_kwh(self):
        return (self.device_tdp_watts * self.elapsed_hours * self.pue) / 1000.0

    def report(self):
        print(f"[EnergyLogger] {self.device_name}: {self.elapsed_hours:.4f}h, {self.energy_kwh:.6f} kWh", flush=True)
        return {"device_name": self.device_name, "elapsed_hours": self.elapsed_hours,
                "device_tdp_watts": self.device_tdp_watts, "pue": self.pue,
                "energy_kwh": self.energy_kwh, "platform": platform.platform()}

    def save(self, path):
        data = self.report()
        data["checkpoints"] = self.checkpoints
        with open(path, "w") as f:
            json.dump(data, f, indent=2)


In [ ]:
# Cell 4b — Persistent text logger (survives disconnects, unlike console/tqdm output)
import datetime

class TextLogger:
    """Appends timestamped lines to a .txt file on Drive so training progress
    is never lost even if the Colab session disconnects and console scrollback
    is gone. Open the file in Drive at any time to check on a running job."""
    def __init__(self, path):
        self.path = path
        with open(self.path, "a") as f:
            f.write(f"\n===== New logging session started {datetime.datetime.now().isoformat()} =====\n")

    def log(self, message):
        line = f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {message}"
        with open(self.path, "a") as f:
            f.write(line + "\n")


In [ ]:
# Cell 5 — Baseline (CoT) dataset + training function
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer, get_linear_schedule_with_warmup

class ReasoningDataset(Dataset):
    def __init__(self, path, tokenizer, max_examples=None, max_length=640):
        self.examples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if max_examples and i >= max_examples:
                    break
                row = json.loads(line)
                text = f"Q: {row['question']}\nReasoning: {row['reasoning']}\nA: {row['answer']}{tokenizer.eos_token}"
                self.examples.append(text)
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.examples[idx], truncation=True, max_length=self.max_length,
                              padding="max_length", return_tensors="pt")
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def train_baseline_cot(data_path, output_dir, ckpt_path, log_path, max_examples=1000, epochs=3,
                        batch_size=8, lr=5e-5, max_length=640, device_name="colab_t4",
                        save_every=500):
    from tqdm.auto import tqdm
    logger_txt = TextLogger(log_path)
    logger_txt.log(f"Starting baseline_cot training: examples={max_examples}, epochs={epochs}, batch_size={batch_size}")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

    dataset = ReasoningDataset(data_path, tokenizer, max_examples, max_length)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    start_step, start_epoch = 0, 0
    elapsed_before = 0.0
    if os.path.exists(ckpt_path):
        print(f"Resuming from checkpoint: {ckpt_path}", flush=True)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_step = ckpt["global_step"]
        start_epoch = ckpt["epoch"]
        elapsed_before = ckpt.get("elapsed_hours", 0.0)
        print(f"Resumed at global_step={start_step}, epoch={start_epoch}", flush=True)
        logger_txt.log(f"RESUMED from checkpoint at global_step={start_step}, epoch={start_epoch}, prior elapsed={elapsed_before:.4f}h")

    logger = EnergyLogger(device_name, REFERENCE_TDP_WATTS[device_name])
    logger.start()
    if elapsed_before:
        logger._start_time -= elapsed_before * 3600.0  # preserve prior elapsed time across resumes

    model.train()
    global_step = start_step
    try:
        for epoch in range(start_epoch, epochs):
            epoch_loss = 0.0
            pbar = tqdm(loader, desc=f"epoch {epoch}")
            for batch in pbar:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                    loss = model(**batch).loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                epoch_loss += loss.item()
                global_step += 1
                pbar.set_postfix(loss=f"{loss.item():.4f}", step=f"{global_step}/{total_steps}")

                if global_step % save_every == 0:
                    torch.save({
                        "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(), "global_step": global_step,
                        "epoch": epoch, "elapsed_hours": logger.elapsed_hours,
                    }, ckpt_path)
                    print(f"  checkpoint saved at step {global_step}", flush=True)
                    logger_txt.log(f"Checkpoint saved: step={global_step}/{total_steps}, loss={loss.item():.4f}")

            print(f"[epoch {epoch}] avg loss: {epoch_loss/len(loader):.4f}", flush=True)
            logger_txt.log(f"Epoch {epoch} complete: avg_loss={epoch_loss/len(loader):.4f}, elapsed={logger.elapsed_hours:.4f}h")
            logger.checkpoint(f"epoch_{epoch}_done")
            torch.save({
                "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(), "global_step": global_step,
                "epoch": epoch + 1, "elapsed_hours": logger.elapsed_hours,
            }, ckpt_path)
    except Exception as e:
        logger_txt.log(f"ERROR during training: {e}")
        logger_txt.log("Progress was checkpointed up to the last save_every interval -- re-run this cell to resume.")
        print(f"ERROR during training: {e}", flush=True)
        print("Progress was checkpointed up to the last save_every interval -- re-run this cell to resume.", flush=True)
        raise

    logger.stop()
    result = logger.report()

    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    logger.save(f"{output_dir}/energy_log.json")

    with open(f"{output_dir}/run_metadata.json", "w") as f:
        json.dump({"method": "baseline_cot", "num_examples": len(dataset), "epochs": epochs,
                    "batch_size": batch_size, "total_steps": total_steps, **result}, f, indent=2)

    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)  # training finished cleanly, checkpoint no longer needed
    logger_txt.log(f"Baseline training COMPLETE: {result['elapsed_hours']:.4f}h, {result['energy_kwh']:.6f} kWh")
    print("Baseline training complete.", flush=True)
    return result


In [ ]:
# Cell 6 — Run baseline training (~10-25 min on T4 for 1000 examples / 3 epochs)
# Progress bar (tqdm) shows live loss/step. Text log at BASE/logs/baseline_log.txt
# survives disconnects. Checkpoints save to Drive every 500 steps -- if this cell
# is interrupted, just re-run it and it will resume automatically.
baseline_result = train_baseline_cot(
    data_path=f"{DATA_DIR}/train_baseline.jsonl",
    output_dir=BASELINE_DIR,
    ckpt_path=f"{CKPT_DIR}/baseline_ckpt.pt",
    log_path=f"{BASE}/logs/baseline_log.txt",
    max_examples=1000,
    epochs=3,
    batch_size=8,
    device_name="colab_t4",
    save_every=500,
)


In [ ]:
# Cell 7 — Coconut dataset, latent-substitution mechanism, and training function
LATENT_TOKEN = "<|latent|>"

class StagedReasoningDataset(Dataset):
    def __init__(self, path, tokenizer, stage, max_examples=None, max_length=640):
        self.rows = []
        with open(path) as f:
            for i, line in enumerate(f):
                if max_examples and i >= max_examples:
                    break
                self.rows.append(json.loads(line))
        self.tokenizer, self.stage, self.max_length = tokenizer, stage, max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        steps = row["reasoning_steps"]
        n_latent = min(self.stage, len(steps))
        latent_part = (LATENT_TOKEN + " ") * n_latent
        text_part = " ".join(steps[n_latent:])
        full_text = f"Q: {row['question']}\nReasoning: {latent_part}{text_part}\nA: {row['answer']}{self.tokenizer.eos_token}"
        enc = self.tokenizer(full_text, truncation=True, max_length=self.max_length,
                              padding="max_length", return_tensors="pt")
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def add_latent_token(tokenizer, model):
    if LATENT_TOKEN not in tokenizer.get_vocab():
        tokenizer.add_special_tokens({"additional_special_tokens": [LATENT_TOKEN]})
        model.resize_token_embeddings(len(tokenizer))
    return tokenizer.convert_tokens_to_ids(LATENT_TOKEN)


def replace_latent_embeddings_with_hidden_state(model, input_ids, inputs_embeds, latent_id):
    """Each <|latent|> position's input embedding is replaced with the model's own
    hidden state from the PRECEDING position (zero-padded shift, not torch.roll --
    roll wraps the last position's state around to position 0, which is wrong)."""
    with torch.no_grad():
        hidden = model.transformer(inputs_embeds=inputs_embeds, output_hidden_states=True).hidden_states[-1]
    latent_mask = (input_ids == latent_id)
    new_embeds = inputs_embeds.clone()
    shifted_hidden = torch.zeros_like(hidden)
    shifted_hidden[:, 1:, :] = hidden[:, :-1, :]
    new_embeds[latent_mask] = shifted_hidden[latent_mask].to(new_embeds.dtype)
    return new_embeds


def train_coconut(data_path, output_dir, ckpt_path, log_path, max_examples=1000, max_latent_stage=2,
                   epochs_per_stage=1, batch_size=8, lr=5e-5, max_length=640,
                   device_name="colab_t4", save_every=500):
    from tqdm.auto import tqdm
    logger_txt = TextLogger(log_path)
    logger_txt.log(f"Starting coconut training: examples={max_examples}, stages={max_latent_stage+1}, epochs_per_stage={epochs_per_stage}, batch_size={batch_size}")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
    latent_id = add_latent_token(tokenizer, model)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    steps_per_epoch = -(-max_examples // batch_size)
    total_steps = steps_per_epoch * epochs_per_stage * (max_latent_stage + 1)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    start_stage, start_epoch, start_step = 0, 0, 0
    global_step = 0
    elapsed_before = 0.0
    if os.path.exists(ckpt_path):
        print(f"Resuming from checkpoint: {ckpt_path}", flush=True)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        global_step = ckpt["global_step"]
        start_stage = ckpt["stage"]
        start_epoch = ckpt["epoch"]
        elapsed_before = ckpt.get("elapsed_hours", 0.0)
        print(f"Resumed at stage={start_stage}, epoch={start_epoch}, global_step={global_step}", flush=True)
        logger_txt.log(f"RESUMED from checkpoint at stage={start_stage}, epoch={start_epoch}, global_step={global_step}, prior elapsed={elapsed_before:.4f}h")

    logger = EnergyLogger(device_name, REFERENCE_TDP_WATTS[device_name])
    logger.start()
    if elapsed_before:
        logger._start_time -= elapsed_before * 3600.0

    try:
        for stage in range(start_stage, max_latent_stage + 1):
            print(f"=== stage {stage}/{max_latent_stage} ===", flush=True)
            dataset = StagedReasoningDataset(data_path, tokenizer, stage, max_examples, max_length)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

            model.train()
            epoch_range = range(start_epoch, epochs_per_stage) if stage == start_stage else range(epochs_per_stage)
            for epoch in epoch_range:
                epoch_loss = 0.0
                pbar = tqdm(loader, desc=f"stage {stage} epoch {epoch}")
                for batch in pbar:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels = batch["labels"].to(device)

                    with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                        inputs_embeds = model.transformer.wte(input_ids)
                        if stage > 0:
                            inputs_embeds = replace_latent_embeddings_with_hidden_state(model, input_ids, inputs_embeds, latent_id)
                        loss = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels).loss

                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()
                    epoch_loss += loss.item()
                    global_step += 1
                    pbar.set_postfix(loss=f"{loss.item():.4f}", step=f"{global_step}/{total_steps}")

                    if global_step % save_every == 0:
                        torch.save({
                            "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                            "scheduler_state": scheduler.state_dict(), "global_step": global_step,
                            "stage": stage, "epoch": epoch, "elapsed_hours": logger.elapsed_hours,
                        }, ckpt_path)
                        print(f"  checkpoint saved at step {global_step}", flush=True)
                        logger_txt.log(f"Checkpoint saved: stage={stage}, step={global_step}/{total_steps}, loss={loss.item():.4f}")

                print(f"  stage {stage} epoch {epoch} avg loss: {epoch_loss/len(loader):.4f}", flush=True)
                logger_txt.log(f"Stage {stage} epoch {epoch} complete: avg_loss={epoch_loss/len(loader):.4f}, elapsed={logger.elapsed_hours:.4f}h")
            start_epoch = 0  # reset for next stage
            logger.checkpoint(f"stage_{stage}_done")
            torch.save({
                "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(), "global_step": global_step,
                "stage": stage + 1, "epoch": 0, "elapsed_hours": logger.elapsed_hours,
            }, ckpt_path)
    except Exception as e:
        logger_txt.log(f"ERROR during training: {e}")
        logger_txt.log("Progress was checkpointed -- re-run this cell to resume.")
        print(f"ERROR during training: {e}", flush=True)
        print("Progress was checkpointed -- re-run this cell to resume.", flush=True)
        raise

    logger.stop()
    result = logger.report()

    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    logger.save(f"{output_dir}/energy_log.json")

    with open(f"{output_dir}/run_metadata.json", "w") as f:
        json.dump({"method": "coconut", "num_examples": max_examples, "max_latent_stage": max_latent_stage,
                    "epochs_per_stage": epochs_per_stage, "total_stages": max_latent_stage + 1,
                    **result}, f, indent=2)

    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    logger_txt.log(f"Coconut training COMPLETE: {result['elapsed_hours']:.4f}h, {result['energy_kwh']:.6f} kWh")
    print("Coconut training complete.", flush=True)
    return result


In [ ]:
# Cell 8 — Run Coconut training (~similar order of time as baseline)
# Same tqdm progress bar, text log (BASE/logs/coconut_log.txt), and 500-step
# checkpointing/resume behavior as the baseline run.
coconut_result = train_coconut(
    data_path=f"{DATA_DIR}/train_coconut.jsonl",
    output_dir=COCONUT_DIR,
    ckpt_path=f"{CKPT_DIR}/coconut_ckpt.pt",
    log_path=f"{BASE}/logs/coconut_log.txt",
    max_examples=1000,
    max_latent_stage=2,
    epochs_per_stage=1,
    batch_size=8,
    device_name="colab_t4",
    save_every=500,
)


In [ ]:
# Cell 9 — Evaluation: accuracy + average generated tokens
def evaluate_model(model_dir, data_path, max_examples=200, max_new_tokens=100,
                    no_repeat_ngram_size=3, repetition_penalty=1.3):
    from tqdm.auto import tqdm
    tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
    model = GPT2LMHeadModel.from_pretrained(model_dir).to(device)
    model.eval()

    rows = []
    with open(data_path) as f:
        for i, line in enumerate(f):
            if i >= max_examples:
                break
            rows.append(json.loads(line))

    correct = 0
    total_tokens = 0
    results_log = []

    for row in tqdm(rows, desc=f"eval {os.path.basename(model_dir)}"):
        prompt = f"Q: {row['question']}\nReasoning:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     pad_token_id=tokenizer.eos_token_id, do_sample=False,
                                     no_repeat_ngram_size=no_repeat_ngram_size,
                                     repetition_penalty=repetition_penalty)

        gen_ids = output[0][prompt_len:]
        n_gen = len(gen_ids)
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

        true_answer = str(row["answer"]).strip().lower()
        is_correct = true_answer in gen_text.strip().lower()

        correct += int(is_correct)
        total_tokens += n_gen
        results_log.append({"question": row["question"], "true_answer": row["answer"],
                             "generated_text": gen_text, "n_generated_tokens": n_gen, "correct": is_correct})

    accuracy = correct / len(rows) if rows else 0.0
    avg_tokens = total_tokens / len(rows) if rows else 0.0
    print(f"RESULTS for {model_dir}\n  Examples: {len(rows)}  Accuracy: {accuracy:.2%}  Avg tokens: {avg_tokens:.1f}", flush=True)

    with open(f"{model_dir}/eval_results.json", "w") as f:
        json.dump({"accuracy": accuracy, "avg_generated_tokens": avg_tokens,
                    "n_examples": len(rows), "per_example": results_log}, f, indent=2)

    return accuracy, avg_tokens


In [ ]:
# Cell 10 — Run evaluation on both models
baseline_accuracy, baseline_avg_tokens = evaluate_model(BASELINE_DIR, f"{DATA_DIR}/eval_baseline.jsonl", max_examples=200)
coconut_accuracy, coconut_avg_tokens = evaluate_model(COCONUT_DIR, f"{DATA_DIR}/eval_coconut.jsonl", max_examples=200)


In [ ]:
# Cell 11 — Break-even calculation and results table
def energy_per_query_kwh(avg_tokens, joules_per_token):
    return (avg_tokens * joules_per_token) / 3_600_000.0

def breakeven_report(baseline_result, coconut_result, baseline_avg_tokens, coconut_avg_tokens, joules_per_token=2.61):
    baseline_kwh = baseline_result["energy_kwh"]
    coconut_kwh = coconut_result["energy_kwh"]
    extra_training_kwh = coconut_kwh - baseline_kwh

    e_baseline = energy_per_query_kwh(baseline_avg_tokens, joules_per_token)
    e_coconut = energy_per_query_kwh(coconut_avg_tokens, joules_per_token)
    savings_per_query = e_baseline - e_coconut

    print("=" * 60)
    print(f"Baseline training: {baseline_kwh:.6f} kWh | Coconut training: {coconut_kwh:.6f} kWh")
    print(f"Extra training cost: {extra_training_kwh:.6f} kWh")
    print(f"Baseline: {baseline_avg_tokens} tok -> {e_baseline:.9f} kWh/query")
    print(f"Coconut:  {coconut_avg_tokens} tok -> {e_coconut:.9f} kWh/query")
    print(f"Savings/query: {savings_per_query:.9f} kWh")

    if extra_training_kwh <= 0 and savings_per_query > 0:
        print("\nCoconut is cheaper on BOTH training and inference -- no break-even needed.")
    elif savings_per_query <= 0:
        print("\nCoconut does NOT save inference energy at these token counts -- no break-even exists.")
    else:
        breakeven = extra_training_kwh / savings_per_query
        print(f"\nBREAK-EVEN: {breakeven:,.0f} queries")
    print("=" * 60)


breakeven_report(baseline_result, coconut_result, baseline_avg_tokens, coconut_avg_tokens)

print("\n\nRESULTS TABLE")
print(f"{'Metric':<35}{'Baseline CoT':<20}{'Coconut':<20}")
print(f"{'Training time (h)':<35}{baseline_result['elapsed_hours']:<20.4f}{coconut_result['elapsed_hours']:<20.4f}")
print(f"{'Training energy (kWh)':<35}{baseline_result['energy_kwh']:<20.6f}{coconut_result['energy_kwh']:<20.6f}")
print(f"{'Accuracy':<35}{baseline_accuracy:<20.2%}{coconut_accuracy:<20.2%}")
print(f"{'Avg tokens/response':<35}{baseline_avg_tokens:<20.1f}{coconut_avg_tokens:<20.1f}")


## How to interpret the results

- **Training energy (kWh)** is an estimate: T4 rated TDP (70W) × wall-clock training
  time, not a live power-sensor reading. State this explicitly as a limitation.
- **Accuracy** at 1,000 examples / 3 epochs should be meaningfully above 0% (unlike
  the 150-example pilot) — if it's still 0%, the model likely needs more epochs or
  examples, not a code fix.
- **Avg tokens/response** compares how many tokens each model generates for its
  final answer on held-out ProsQA questions — lower for Coconut supports its core
  efficiency claim, if reasoning quality (accuracy) held up.
- **Break-even**: if Coconut's training energy is *higher* than baseline's, the
  break-even number tells you how many inference queries are needed before
  Coconut's smaller per-query cost pays back the extra training energy. If
  Coconut's training energy is *lower*, there's no break-even needed — it wins
  on both axes immediately (as happened in the 150-example pilot run).
- All checkpoints, models, and results are saved under
  `/content/drive/MyDrive/coconut_research_scaled/` — nothing is lost if this
  notebook disconnects; re-running a training cell resumes from the last save.
